<a href="https://colab.research.google.com/github/KalinaMarkova/deep_learning_course_project/blob/main/03_Model_Training_Experiment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Grained Analysis of Propaganda in News Articles
## Notebook 03: Model Training for End-to-End Joint Tagger (29 Labels) - Experiment 1

In this notebook, we will load our tokenized and aligned dataset and use it to train a **RoBERTa** model to identify the exact boundaries of propaganda techniques by Token Classification.

We are going to add a brand-new, untrained mathematical layer directly on top of RoBERTa which will be the Token Classifier. As RoBERTa reads an article, it will push its contextual understanding of every single token up into that classification head. The head then will produce a prediction for every single token, trying to guess which of the integer labels (0 for non-propaganda and labels 1 through 28 for the specific propaganda categories) belongs to that token.

In [2]:
!pip install evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=67f31a9c675ffb9854e9f2aa6f90aeff27ac873138e0bc695342ccd2182c3d88
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [22]:
import torch
import numpy as np
import os
import evaluate
from google.colab import drive
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments
)
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from seqeval.metrics import classification_report
from collections import Counter
import shutil



Let's load the dataset, split it and recreate again 29-class Label Dictionaries.

In [5]:
drive.mount('/content/drive')

# 1. Load the dataset from your correct Drive path
dataset_path = '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/exp1_joint_29labels_sentence_dataset'
dataset = load_from_disk(dataset_path)

# 2. Perform the Three-Way Split (80% Train, 10% Val, 10% Test)
# First, separate 80% for training and 20% for the temporary hold-out
train_temp_split = dataset.train_test_split(test_size=0.20, seed=42)
train_dataset = train_temp_split['train']
temp_dataset = train_temp_split['test']

# Next, split that 20% hold-out evenly into 10% Validation and 10% Test
val_test_split = temp_dataset.train_test_split(test_size=0.50, seed=42)
val_dataset = val_test_split['train']
test_dataset = val_test_split['test']

# 3. Recreate the 29-class Label Dictionaries perfectly
semeval_techniques = [
    "Appeal_to_Authority", "Appeal_to_fear-prejudice", "Bandwagon,Reductio_ad_hitlerum",
    "Black-and-White_Fallacy", "Causal_Oversimplification", "Doubt",
    "Exaggeration,Minimisation", "Flag-Waving", "Loaded_Language",
    "Name_Calling,Labeling", "Repetition", "Slogans",
    "Thought-terminating_Cliches", "Whataboutism,Straw_Man,Red_Herring"
]

exp1_labels_list = ['O'] + [f"B-{t}" for t in semeval_techniques] + [f"I-{t}" for t in semeval_techniques]
exp1_labels_list = sorted(list(set(exp1_labels_list)))
exp1_labels_list.remove('O')
exp1_labels_list = ['O'] + exp1_labels_list

label2id = {label: i for i, label in enumerate(exp1_labels_list)}
id2label = {i: label for label, i in label2id.items()}

print(f"Dataset split is completed.")
print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

Mounted at /content/drive
Dataset split is completed.
Train size: 12022
Validation size: 1503
Test size: 1503


In order to manage the extreme class imbalance of non-propaganda text vs. the different types of propaganda and to avoid the model just guessing "O" for every word, we will use the `compute_class_weight` for the loss function of the model which mathematically penalizes the model for missing rare classes using the following Inverse Frequency equation:

$$W_j = \frac{N}{k \times n_j}$$

**Where:**
* $W_j$ = The final computed weight for class $j$
* $N$ = The total number of tokens (samples) in your entire training dataset
* $k$ = The total number of unique classes (e.g., 29)
* $n_j$ = The exact number of times class $j$ actually appears in the dataset



In [6]:
# Extract all labels from the training set and flatten them into a single list
all_train_labels = [label for sequence in train_dataset['labels'] for label in sequence]

# Identify unique classes present in the training set
unique_classes = np.unique(all_train_labels)

# Compute balanced class weights
weights = compute_class_weight(class_weight='balanced', classes=unique_classes, y=all_train_labels)

# Create a full weight array for all 29 labels (defaulting to 1.0 if a class is completely missing)
class_weights = np.ones(len(label2id))
for i, cls in enumerate(unique_classes):
    class_weights[cls] = weights[i]

# Convert to a PyTorch Tensor and move it to your GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print(f"Class weights successfully calculated for {len(unique_classes)} unique classes!")
print(f"Hardware in use: {device}")

Class weights successfully calculated for 29 unique classes!
Hardware in use: cuda


RoBERTa was originally trained to fill in missing words. By using `AutoModelForTokenClassification`, Hugging Face removes RoBERTa's original "guessing" head and glues on a brand-new, completely blank neural network layer designed specifically to output one of your 29 labels (`num_labels`). When `num_labels = 29`, this projection compresses the 768 complex features representing a token down into **29 raw numbers**—one scalar confidence score for every category in your label dictionary.

For example, the output array of raw logits for the word *"fake"* might look like this:

| Label Index | Label String (`id2label`) | Raw Logit Score (Model Confidence) |
| :--- | :--- | :--- |
| `0` | `"O"` | `-1.2` |
| `1` | `"B-Loaded_Language"` | **`+8.5`** $\leftarrow$ *(Highest Score)* |
| `2` | `"I-Loaded_Language"` | `+0.3` |
| `3` | `"B-Name_Calling,Labeling"` | `+1.1` |
| `...` | `...` | `...` |
| `28` | `"I-Slogans"` | `-3.4` |



The official SemEval competition used a custom scoring script that awarded **partial credit** if a model's prediction overlapped with the true propaganda span (e.g., predicting "radical left" when the true label was just "radical").

However, during the training phase, we will use **`seqeval`** which requires exact match with the golden start to give credit to the model. Training under these harsh, strict rules forces the model to learn highly precise boundaries. `seqeval` is the industry standard for sequence tagging. It is well integrated into the Hugging Face `Trainer` and calculates metrics instantly at the end of every epoch, whereas a custom character-overlap script would massively slow down training. It is standard practice to train and validate using a fast, strict metric like `seqeval`. Once training is completely finished, we can take the final model and run the official SemEval partial-credit script on the Test Set to get the "official" publication-ready score.


RoBERTa was originally trained to fill in missing words. By using `AutoModelForTokenClassification`, Hugging Face removes RoBERTa's original "guessing" head and glues on a brand-new, completely blank neural network layer designed specifically to output one of your 29 labels (`num_labels`). When `num_labels = 29`, this projection compresses the 768 complex features representing a token down into **29 raw numbers**—one scalar confidence score for every category in your label dictionary.

For example, the output array of raw logits for the word *"fake"* might look like this:

| Label Index | Label String (`id2label`) | Raw Logit Score (Model Confidence) |
| :--- | :--- | :--- |
| `0` | `"O"` | `-1.2` |
| `1` | `"B-Loaded_Language"` | **`+8.5`** $\leftarrow$ *(Highest Score)* |
| `2` | `"I-Loaded_Language"` | `+0.3` |
| `3` | `"B-Name_Calling,Labeling"` | `+1.1` |
| `...` | `...` | `...` |
| `28` | `"I-Slogans"` | `-3.4` |



The official SemEval competition used a custom scoring script that awarded **partial credit** if a model's prediction overlapped with the true propaganda span (e.g., predicting "radical left" when the true label was just "radical").

However, during the training phase, we will use **`seqeval`** which requires exact match with the golden start to give credit to the model. Training under these harsh, strict rules forces the model to learn highly precise boundaries. `seqeval` is the industry standard for sequence tagging. It is well integrated into the Hugging Face `Trainer` and calculates metrics instantly at the end of every epoch, whereas a custom character-overlap script would massively slow down training. Once training is completely finished, we can take the final model and run the official SemEval partial-credit script on the Test Set to get the "official" publication-ready score.


In [7]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 1. Pop the true labels out of the inputs
        labels = inputs.pop("labels")

        # 2. Feed the text to RoBERTa to get its predictions (logits)
        outputs = model(**inputs)
        logits = outputs.logits

        # 3. Define our Custom Loss Function using our class weights
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)

        # 4. Flatten everything to align it mathematically
        # Use -100 to find and ignore all padding tokens
        active_loss = labels.view(-1) != -100
        active_logits = logits.view(-1, self.model.config.num_labels)[active_loss]
        active_labels = labels.view(-1)[active_loss]

        # 5. Calculate the weighted loss
        loss = loss_fct(active_logits, active_labels)

        return (loss, outputs) if return_outputs else loss

# Initialize seqeval metric
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    # Convert probability scores into exact class predictions
    predictions = np.argmax(predictions, axis=2)

    # Remove the padding (-100) and convert Integer IDs back to String Labels (e.g., 'B-Slogans')
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Let seqeval calculate F1 based on entire BIO spans, not just individual words
    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

The Hyperparameters:

`learning_rate=2e-5 (0.00002)`: Because RoBERTa is already incredibly smart, we want to update its knowledge gently. If the learning rate is too high, the model "forgets" the English language. 2e-5 is the gold standard for fine-tuning NLP models.

`per_device_train_batch_size=16`: The model will look at 16 sentences at a time, calculate its errors, and update its weights. This size is sutable for the capabilities of Google Colab.

`num_train_epochs=3`: The model will read the entire training dataset front-to-back exactly 3 times. For complex tasks like 29-label sequence tagging, going beyond 3 or 4 epochs usually leads to overfitting, which means the model is memorizing the training data instead of learning the actual patterns.

`weight_decay=0.01`: A regularization technique that slightly shrinks the model's weights during training. This prevents the model from memorizing the training data (overfitting) and helps it generalize better to unseen text.

`eval_strategy & save_strategy="epoch"`: At the end of every epoch, the model pauses, takes a test on the 10% Validation set, prints the F1 score, and saves a checkpoint of its brain to your Google Drive.

`load_best_model_at_end=True & metric_for_best_model="f1"`: Often, a model might peak at Epoch 2 and actually get worse in Epoch 3. This setting ensures that when training finishes, Hugging Face deletes the inferior versions and keeps the checkpoint that achieved the highest F1 score on the validation data.

`weight_decay=0.01`: A regularization technique that slightly shrinks the model's weights during training. This prevents the model from memorizing the training data (overfitting) and helps it generalize better to unseen text.

In [11]:
# 1. Initialize Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForTokenClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
).to(device)

# 2. Data Collator for dynamic padding
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 3. Setup output directory in your Google Drive to save the trained model
output_directory = '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/results_exp1'
os.makedirs(output_directory, exist_ok=True)

# 4. Define Training Hyperparameters
training_args = TrainingArguments(
    output_dir=output_directory,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

# 5. Initialize our Custom Trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# 6. Start the training
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,2.384492,2.418637,0.004154,0.090136,0.007942,0.414138
2,1.867370,2.064039,0.007002,0.137755,0.013327,0.447628
3,1.471347,2.045089,0.007095,0.130952,0.013460,0.505286


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2256, training_loss=2.0422054537644625, metrics={'train_runtime': 706.9998, 'train_samples_per_second': 51.013, 'train_steps_per_second': 3.191, 'total_flos': 1618921298721804.0, 'train_loss': 2.0422054537644625, 'epoch': 3.0})

The Training Loss is dropping consistently from 2.32 to 1.86 to 1.53. The Validation Loss is also dropping (2.35 down to 2.02), which proves that the model is learnig although very slowly.

Although the Accuracy seems ok at around 52.6%, the F1 score is 0.013. As Accuracy includes the "O" tags and because the vast majority of the text is not propaganda, the model can achieve a 50%+ accuracy simply by guessing "O" for almost every word. `seqeval`, however, completely ignores the "O" tag when calculating F1. It only scores the model on how well it identifies the actual B- and I- propaganda tags.

The Precision is 0.007 (0.7%), but the Recall is 0.127 (12.7%) due to the  Inverse Frequency weighting as the model is penalized the model so heavily for missing rare classes, it is actually "over-guessing" propaganda tags to avoid the penalty.


Let's generate a detailed classification report so we can see exactly which of the 14 propaganda techniques the model is failing on the hardest.

In [12]:
# 1. Ask the Trainer to predict on the test_dataset
predictions_output = trainer.predict(test_dataset)

# 2. Extract the raw numerical predictions and true labels
predictions = np.argmax(predictions_output.predictions, axis=2)
true_labels = predictions_output.label_ids

# 3. Clean up the padding (-100) and convert Integer IDs back to Strings
true_predictions = [
    [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, true_labels)
]
true_references = [
    [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, true_labels)
]

# 4. Generate and print the detailed classification report
print("\n" + "="*50)
print(" FINAL TEST SET CLASSIFICATION REPORT")
print("="*50)
report = classification_report(true_references, true_predictions)
print(report)


 FINAL TEST SET CLASSIFICATION REPORT
                                    precision    recall  f1-score   support

               Appeal_to_Authority       0.00      0.00      0.00        13
          Appeal_to_fear-prejudice       0.00      0.00      0.00        27
    Bandwagon,Reductio_ad_hitlerum       0.00      0.00      0.00         4
           Black-and-White_Fallacy       0.00      0.14      0.01         7
         Causal_Oversimplification       0.00      0.04      0.00        28
                             Doubt       0.00      0.08      0.00        39
         Exaggeration,Minimisation       0.01      0.17      0.02        46
                       Flag-Waving       0.00      0.05      0.00        21
                   Loaded_Language       0.02      0.18      0.04       214
             Name_Calling,Labeling       0.02      0.18      0.04       100
                        Repetition       0.01      0.22      0.02        69
                           Slogans       0.00   


We can see that Loaded_Language appeared 214 times in the test dataset, and Name_Calling,Labeling appeared 100 times. At the same time, Bandwagon appeared only 4 times, and Slogans only 6 times, proving that RoBERTa simply did not see enough examples of these rare classes to map out their complex linguistic patterns.

We see that Repetition has a Recall of 0.22 (22%) but a Precision of 0.01 (1%). This proves that the class weights worked, but probably were too strong. The mathematical penalty was so high that RoBERTa got terrified of missing these classes. To avoid the penalty, it started desperately throwing the Repetition and Cliche tags at completely random words. Because it guessed those tags so often, it accidentally caught about 20-25% of the true ones (Recall), but because 99% of its guesses were wild misses, the Precision plummeted to 1%.

The Micro Average F1 is 0.02 (2%). As discussed earlier, `seqeval` demands absolute perfection. If the human annotator tagged a 5-word phrase as Loaded_Language, and RoBERTa only tagged 4 of those words, RoBERTa gets a 0.00 for that prediction. Given the subjective nature of propaganda, asking the model to hit the exact human boundaries on a 14-class task after only 3 epochs is really too much to ask.

Now let's calculate how many times the model predicts each label. Calculating the prediction frequencies will immediately tell us if the weights were too aggressive.

In [14]:
# 1. Flatten the lists of lists into single long lists
flat_predictions = [tag for sentence in true_predictions for tag in sentence]
flat_references = [tag for sentence in true_references for tag in sentence]

# 2. Count the occurrences of each tag
pred_counts = Counter(flat_predictions)
true_counts = Counter(flat_references)

# 3. Print a side-by-side comparison (Skipping 'O' to focus on propaganda)
print(f"{'Propaganda Tag':<45} | {'True Count':<12} | {'Predicted Count'}")
print("-" * 75)

# Sort alphabetically for easy reading
for tag in sorted(label2id.keys()):
    if tag == 'O':
        continue

    t_count = true_counts.get(tag, 0)
    p_count = pred_counts.get(tag, 0)

    # Highlight extreme differences with a marker
    marker = " <--- ALARM" if p_count > (t_count * 5) and p_count > 50 else ""

    print(f"{tag:<45} | {t_count:<12} | {p_count}{marker}")

print("\n" + "-" * 75)
print(f"Total 'O' (Background) True Counts: {true_counts.get('O', 0)}")
print(f"Total 'O' (Background) Pred Counts: {pred_counts.get('O', 0)}")

Propaganda Tag                                | True Count   | Predicted Count
---------------------------------------------------------------------------
B-Appeal_to_Authority                         | 12           | 1053 <--- ALARM
B-Appeal_to_fear-prejudice                    | 26           | 592 <--- ALARM
B-Bandwagon,Reductio_ad_hitlerum              | 4            | 53 <--- ALARM
B-Black-and-White_Fallacy                     | 7            | 141 <--- ALARM
B-Causal_Oversimplification                   | 22           | 569 <--- ALARM
B-Doubt                                       | 38           | 954 <--- ALARM
B-Exaggeration,Minimisation                   | 41           | 619 <--- ALARM
B-Flag-Waving                                 | 19           | 292 <--- ALARM
B-Loaded_Language                             | 213          | 1540 <--- ALARM
B-Name_Calling,Labeling                       | 97           | 665 <--- ALARM
B-Repetition                                  | 67           | 9

This confirms that the model heavily overcompensated for the class weights.

The model effectively took ~20,000 normal, background words and hallucinated that they were propaganda because the penalty for the "O" class was so tiny (around 0.03), while the penalty for missing a propaganda tag was massive (up to 34+).

In addition, because B- tags are  rarer than I- tags (a phrase only has one B- but can have many I-s), the scikit-learn balanced weight calculation gave the B- tags the highest penalty multipliers. The model reacted by spamming B- tags everywhere just to avoid the penalty.

Before we try to mathematically fix the imbalance with smoothed weights, let's see what the model naturally does when left to its own devices with no class weights.

In [15]:
# 1. Initialize a new model
unweighted_model = AutoModelForTokenClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# 2. Define fresh training arguments (Note the new output_dir!)
unweighted_training_args = TrainingArguments(
    output_dir="./unweighted_roberta_propaganda",  # Kept separate from Experiment 1
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

# 3. Initialize the standard Hugging Face Trainer (No class weights!)
unweighted_trainer = Trainer(
    model=unweighted_model,
    args=unweighted_training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# 4. Start the new training loop
print("Starting Unweighted Baseline Training...")
unweighted_trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting Unweighted Baseline Training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.510767,0.505932,0.097826,0.045918,0.062500,0.891544
2,0.466151,0.469064,0.105839,0.049320,0.067285,0.895527
3,0.303024,0.483788,0.099783,0.078231,0.087703,0.892232


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


...

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2256, training_loss=0.44126046327411705, metrics={'train_runtime': 628.9489, 'train_samples_per_second': 57.343, 'train_steps_per_second': 3.587, 'total_flos': 1618921298721804.0, 'train_loss': 0.44126046327411705, 'epoch': 3.0})


Now we see the exact opposite behavior of compared to the first model. Accuracy us ~89.2%. Without the massive penalties of the class weights, the model quickly realized that 90% of the dataset is just normal background text. By safely guessing "O" for almost everything, it easily achieved a high accuracy score.

This model only guessed propaganda when it was highly confident. Because it made fewer guesses, its Precision shot up to ~10%. However, because it was so timid, it missed a lot of actual propaganda, keeping its Recall low at ~7.8%.

In Epoch 3, the Training Loss drops significantly (down to 0.30), but the Validation Loss actually rises slightly from 0.469 to 0.483, signifying
overfitting. By the third epoch, RoBERTa started memorizing the specific sentences in the training data rather than learning general rules, which caused its performance on the unseen validation data to degrade slightly.



In [16]:

# --- SEQEVAL CLASSIFICATION REPORT ---
# 1. Ask the Unweighted Trainer to predict on the test_dataset
unweighted_predictions_output = unweighted_trainer.predict(test_dataset)

# 2. Extract the raw numerical predictions and true labels
unweighted_predictions = np.argmax(unweighted_predictions_output.predictions, axis=2)
unweighted_true_labels = unweighted_predictions_output.label_ids

# 3. Clean up the padding (-100) and convert Integer IDs back to Strings
unweighted_true_predictions = [
    [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(unweighted_predictions, unweighted_true_labels)
]
unweighted_true_references = [
    [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(unweighted_predictions, unweighted_true_labels)
]

# 4. Generate and print the detailed classification report
print("="*60)
print(" UNWEIGHTED MODEL: FINAL TEST SET CLASSIFICATION REPORT")
print("="*60)
unweighted_report = classification_report(unweighted_true_references, unweighted_true_predictions)
print(unweighted_report)


# --- PREDICTION FREQUENCY ANALYZER ---


# 1. Flatten the lists of lists into single long lists
unweighted_flat_predictions = [tag for sentence in unweighted_true_predictions for tag in sentence]
unweighted_flat_references = [tag for sentence in unweighted_true_references for tag in sentence]

# 2. Count the occurrences of each tag
unweighted_pred_counts = Counter(unweighted_flat_predictions)
unweighted_true_counts = Counter(unweighted_flat_references)

# 3. Print a side-by-side comparison
print(f"{'Propaganda Tag':<45} | {'True Count':<12} | {'Predicted Count (Unweighted)'}")
print("-" * 85)

# Sort alphabetically for easy reading
for tag in sorted(label2id.keys()):
    if tag == 'O':
        continue

    t_count = unweighted_true_counts.get(tag, 0)
    p_count = unweighted_pred_counts.get(tag, 0)

    # Highlight extreme differences with a marker
    marker = " <--- ALARM (Over-predicting)" if p_count > (t_count * 5) and p_count > 50 else ""
    # Highlight ignored classes (typical for unweighted models)
    zero_marker = " <--- IGNORING (0 Predictions)" if p_count == 0 and t_count > 0 else ""

    print(f"{tag:<45} | {t_count:<12} | {p_count}{marker}{zero_marker}")

print("\n" + "-" * 85)
print(f"Total 'O' (Background) True Counts: {unweighted_true_counts.get('O', 0)}")
print(f"Total 'O' (Background) Pred Counts: {unweighted_pred_counts.get('O', 0)}")

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


 UNWEIGHTED MODEL: FINAL TEST SET CLASSIFICATION REPORT
                                    precision    recall  f1-score   support

               Appeal_to_Authority       0.00      0.00      0.00        13
          Appeal_to_fear-prejudice       0.00      0.00      0.00        27
    Bandwagon,Reductio_ad_hitlerum       0.00      0.00      0.00         4
           Black-and-White_Fallacy       0.00      0.00      0.00         7
         Causal_Oversimplification       0.00      0.00      0.00        28
                             Doubt       0.03      0.03      0.03        39
         Exaggeration,Minimisation       0.00      0.00      0.00        46
                       Flag-Waving       0.05      0.05      0.05        21
                   Loaded_Language       0.13      0.14      0.13       214
             Name_Calling,Labeling       0.09      0.11      0.10       100
                        Repetition       0.00      0.00      0.00        69
                           Slog

Exactly as expected, the unweighted model completely gave up on 10 out of the 14 propaganda techniques. Classes like Bandwagon, Slogans, and Repetition all have flat 0.00 scores. Without the  penalty of the class weights forcing it to care, RoBERTa realized these classes were so rare that missing them barely affected its overall loss score, so it just ignored them entirely.

Because Loaded_Language and Name_Calling made up the vast majority of the propaganda examples in the training data, the model dedicated all of its learning capacity to those two specific techniques and ignored the rest.

In addition, it predicted exactly 0 for every single B- (Beginning) tag. `seqeval` relies on strict BIO logic. To score a point, the model must predict a B- tag to signal the start of a propaganda span. Because the unweighted model refused to predict a single B- tag, it essentially never started a valid prediction.

Now let's run a model with smoothed weights. We will use the raw mathematical weights just like inthe first model, but this time we will apply a square root function (np.sqrt) to "smooth" them out. This will compress the massive 50x penalties down to a much more reasonable scale.


$$w_{\text{smoothed}\_c} = \sqrt{\frac{N}{C \cdot n_c}}$$



In [17]:
# 1. Extract all active labels and calculate raw weights
train_labels = [label for feature in train_dataset for label in feature['labels'] if label != -100]
raw_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

# 2.  Take the square root of the raw weights
smoothed_weights = np.sqrt(raw_weights)

# 3. overwrite the variable the existing WeightedTrainer uses
device = "cuda" if torch.cuda.is_available() else "cpu"
class_weights_tensor = torch.tensor(smoothed_weights, dtype=torch.float32).to(device)
print("Updated class_weights_tensor with smoothed values.")

# 4. Initialize a new model
smoothed_model = AutoModelForTokenClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# 5. Define Training Arguments
smoothed_training_args = TrainingArguments(
    output_dir="./smoothed_roberta_propaganda",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

# 6. Reuse the exact WeightedTrainer class
smoothed_trainer = WeightedTrainer(
    model=smoothed_model,
    args=smoothed_training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# 7. Start the training loop
smoothed_trainer.train()

Updated class_weights_tensor with smoothed values.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.754349,1.854148,0.018104,0.100340,0.030673,0.783922
2,1.564074,1.627808,0.030690,0.151361,0.051032,0.801501
3,0.997136,1.676665,0.034828,0.159864,0.057195,0.818371


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2256, training_loss=1.4970846979330616, metrics={'train_runtime': 681.2918, 'train_samples_per_second': 52.938, 'train_steps_per_second': 3.311, 'total_flos': 1618921298721804.0, 'train_loss': 1.4970846979330616, 'epoch': 3.0})

This third model achieved, the highest Recall out of all three models. This means smoothing the weights pushed the model to look for rare classes. Precision also improved to 3.4%, although still very low.

However, just like in the second model, there is overfitting after Epoch 2, when validation Loss increased.

In [18]:
# --- SEQEVAL CLASSIFICATION REPORT ---

# 1. Ask the Smoothed Trainer to predict on the test_dataset
smoothed_predictions_output = smoothed_trainer.predict(test_dataset)

# 2. Extract the raw numerical predictions and true labels
smoothed_predictions = np.argmax(smoothed_predictions_output.predictions, axis=2)
smoothed_true_labels = smoothed_predictions_output.label_ids

# 3. Clean up the padding (-100) and convert Integer IDs back to Strings
smoothed_true_predictions = [
    [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(smoothed_predictions, smoothed_true_labels)
]
smoothed_true_references = [
    [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(smoothed_predictions, smoothed_true_labels)
]

# 4. Generate and print the detailed classification report
print("="*60)
print(" SMOOTHED MODEL: FINAL TEST SET CLASSIFICATION REPORT")
print("="*60)
smoothed_report = classification_report(smoothed_true_references, smoothed_true_predictions)
print(smoothed_report)


# --- PREDICTION FREQUENCY ANALYZER ---


# 1. Flatten the lists of lists into single long lists
smoothed_flat_predictions = [tag for sentence in smoothed_true_predictions for tag in sentence]
smoothed_flat_references = [tag for sentence in smoothed_true_references for tag in sentence]

# 2. Count the occurrences of each tag
smoothed_pred_counts = Counter(smoothed_flat_predictions)
smoothed_true_counts = Counter(smoothed_flat_references)

# 3. Print a side-by-side comparison
print(f"{'Propaganda Tag':<45} | {'True Count':<12} | {'Predicted Count (Smoothed)'}")
print("-" * 85)

# Sort alphabetically for easy reading
for tag in sorted(label2id.keys()):
    if tag == 'O':
        continue

    t_count = smoothed_true_counts.get(tag, 0)
    p_count = smoothed_pred_counts.get(tag, 0)

    # Let's see if it still panics or ignores!
    marker = ""
    if p_count > (t_count * 5) and p_count > 50:
        marker = " <--- ALARM (Over-predicting)"
    elif p_count == 0 and t_count > 0:
        marker = " <--- IGNORING (0 Predictions)"

    print(f"{tag:<45} | {t_count:<12} | {p_count}{marker}")

print("\n" + "-" * 85)
print(f"Total 'O' (Background) True Counts: {smoothed_true_counts.get('O', 0)}")
print(f"Total 'O' (Background) Pred Counts: {smoothed_pred_counts.get('O', 0)}")

 SMOOTHED MODEL: FINAL TEST SET CLASSIFICATION REPORT
                                    precision    recall  f1-score   support

               Appeal_to_Authority       0.00      0.00      0.00        13
          Appeal_to_fear-prejudice       0.00      0.00      0.00        27
    Bandwagon,Reductio_ad_hitlerum       0.00      0.00      0.00         4
           Black-and-White_Fallacy       0.00      0.00      0.00         7
         Causal_Oversimplification       0.00      0.00      0.00        28
                             Doubt       0.01      0.05      0.01        39
         Exaggeration,Minimisation       0.02      0.11      0.04        46
                       Flag-Waving       0.03      0.14      0.05        21
                   Loaded_Language       0.06      0.24      0.10       214
             Name_Calling,Labeling       0.04      0.17      0.07       100
                        Repetition       0.03      0.04      0.03        69
                           Slogan

In [20]:
def extract_spans(tags):
    """Converts a list of BIO tags into spans: (label, start_index, end_index)"""
    spans = []
    current_span = None

    for i, tag in enumerate(tags):
        if tag == 'O':
            if current_span:
                spans.append(current_span)
                current_span = None
        elif tag.startswith('B-'):
            if current_span:
                spans.append(current_span)
            current_span = (tag[2:], i, i)
        elif tag.startswith('I-'):
            if current_span and current_span[0] == tag[2:]:
                # Extend the current span
                current_span = (current_span[0], current_span[1], i)
            else:
                # Malformed I-tag (starts without a B-tag)
                if current_span:
                    spans.append(current_span)
                current_span = (tag[2:], i, i)

    if current_span:
        spans.append(current_span)
    return spans

total_true_spans = 0
total_pred_spans = 0
total_partial_recall_score = 0.0
total_partial_precision_score = 0.0

# Evaluate sentence by sentence
for true_tags, pred_tags in zip(smoothed_true_references, smoothed_true_predictions):
    true_spans = extract_spans(true_tags)
    pred_spans = extract_spans(pred_tags)

    total_true_spans += len(true_spans)
    total_pred_spans += len(pred_spans)

    # 1. Calculate Partial Recall
    for t_label, t_start, t_end in true_spans:
        t_length = t_end - t_start + 1
        best_overlap = 0

        for p_label, p_start, p_end in pred_spans:
            if t_label == p_label:
                # Calculate overlap between ranges
                overlap_start = max(t_start, p_start)
                overlap_end = min(t_end, p_end)
                if overlap_start <= overlap_end:
                    overlap_len = overlap_end - overlap_start + 1
                    best_overlap = max(best_overlap, overlap_len)

        total_partial_recall_score += (best_overlap / t_length)

    # 2. Calculate Partial Precision
    for p_label, p_start, p_end in pred_spans:
        p_length = p_end - p_start + 1
        best_overlap = 0

        for t_label, t_start, t_end in true_spans:
            if p_label == t_label:
                overlap_start = max(p_start, t_start)
                overlap_end = min(p_end, t_end)
                if overlap_start <= overlap_end:
                    overlap_len = overlap_end - overlap_start + 1
                    best_overlap = max(best_overlap, overlap_len)

        total_partial_precision_score += (best_overlap / p_length)

# Calculate final percentages
partial_precision = total_partial_precision_score / total_pred_spans if total_pred_spans > 0 else 0
partial_recall = total_partial_recall_score / total_true_spans if total_true_spans > 0 else 0

if (partial_precision + partial_recall) > 0:
    partial_f1 = 2 * (partial_precision * partial_recall) / (partial_precision + partial_recall)
else:
    partial_f1 = 0.0

print("="*60)
print(" SEMEVAL-STYLE PARTIAL OVERLAP SCORES (Smoothed Model)")
print("="*60)
print(f"Exact-Match F1 (For Context) : ~0.060 (6.0%)")
print("-" * 60)
print(f"Total Gold Spans (Ground Truth) : {total_true_spans}")
print(f"Total Predicted Spans (Model)  : {total_pred_spans}")
print("-" * 60)
print(f"Partial Precision               : {partial_precision:.4f} ({partial_precision*100:.1f}%)")
print(f"Partial Recall                  : {partial_recall:.4f} ({partial_recall*100:.1f}%)")
print(f"Partial F1 Score                : {partial_f1:.4f} ({partial_f1*100:.1f}%)")
print("="*60)

 SEMEVAL-STYLE PARTIAL OVERLAP SCORES (Smoothed Model)
Exact-Match F1 (For Context) : ~0.060 (6.0%)
------------------------------------------------------------
Total Gold Spans (Ground Truth) : 587
Total Predicted Spans (Model)  : 2519
------------------------------------------------------------
Partial Precision               : 0.1604 (16.0%)
Partial Recall                  : 0.4463 (44.6%)
Partial F1 Score                : 0.2360 (23.6%)


The jump from ~6% Exact F1 to 23.6% Partial F1 with a 44.6% Partial Recall confirms that the model is actually finding nearly half of all propaganda spans.

Let's save the model, so we can use it in the future if needed.

In [1]:
drive_save_directory = "/content/drive/MyDrive/Fine_Grained_Propaganda_Analysis"
os.makedirs(drive_save_directory, exist_ok=True)

# 1. Save model weights & config
smoothed_model.save_pretrained(drive_save_directory)

# 2. Save tokenizer
tokenizer.save_pretrained(drive_save_directory)

NameError: name 'os' is not defined